Cell 1 — Mount Google Drive

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive
# Experiment: MMS-Tarifit fine-tuning on Corpus V1.1
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


Cell 2 — Define project paths

In [ ]:
# ============================================================
# CELL 2: Define MMS fine-tuning paths
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

TRAIN_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1_1"
    / "train.csv"
)

VALIDATION_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1_1"
    / "validation.csv"
)

MMS_VOCAB_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mms_tokenizer_v1_1"
)

MMS_VOCAB_PATH = (
    MMS_VOCAB_DIR
    / "vocab.json"
)

MMS_DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mms_corpus_v1_1"
)

MMS_OUTPUT_DIR = (
    PROJECT_ROOT
    / "models"
    / "mms_tarifit_finetuned_v1_1"
)

print("Project:", PROJECT_ROOT.exists())
print("Train:", TRAIN_CSV.exists())
print("Validation:", VALIDATION_CSV.exists())

Project: True
Train: True
Validation: True


Cell 3 — Install dependencies

In [ ]:
# ============================================================
# CELL 3: Install MMS fine-tuning dependencies
# ============================================================

!pip install -q transformers datasets accelerate jiwer soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.1 MB/s eta 0:00:00


Cell 4 — Load Corpus V1.1

In [ ]:
# ============================================================
# CELL 4: Load Corpus V1.1
# ============================================================

import pandas as pd
from datasets import Dataset, DatasetDict

train_df = pd.read_csv(TRAIN_CSV)
validation_df = pd.read_csv(VALIDATION_CSV)

dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df,
        preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        validation_df,
        preserve_index=False
    ),
})

print(dataset)

print(
    "Train hours:",
    round(
        train_df["duration_seconds"].sum() / 3600,
        3
    )
)

print(
    "Validation minutes:",
    round(
        validation_df["duration_seconds"].sum() / 60,
        2
    )
)

DatasetDict({
    train: Dataset({
        features: ['segment_id', 'recording_id', 'speaker_group_id', 'audio_path', 'duration_seconds', 'transcription', 'absolute_audio_path'],
        num_rows: 1472
    })
    validation: Dataset({
        features: ['segment_id', 'recording_id', 'speaker_group_id', 'audio_path', 'duration_seconds', 'transcription', 'absolute_audio_path'],
        num_rows: 133
    })
})
Train hours: 4.638
Validation minutes: 18.12


Cell 5 — Build Corpus V1.1 CTC vocabulary

In [ ]:
# ============================================================
# CELL 5: Build Corpus V1.1 MMS CTC vocabulary
# ============================================================

import json

all_text = (
    train_df["transcription"].astype(str).tolist()
    + validation_df["transcription"].astype(str).tolist()
)

characters = sorted(
    set("".join(all_text))
)

print("Corpus characters:")
print(characters)

print("\nř present:", "ř" in characters)

vocab_dict = {
    char: idx
    for idx, char in enumerate(characters)
}

# Space -> CTC word separator
space_id = vocab_dict.pop(" ")
vocab_dict["|"] = space_id

# Special tokens
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print("\nVocabulary size:", len(vocab_dict))

Corpus characters:
[' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'p', 'q', 'r', 's', 't', 'u', 'w', 'x', 'y', 'z', 'ǧ', 'ɛ', 'ɣ', 'ʷ', 'ḍ', 'ḥ', 'ṣ', 'ṭ', 'ẓ']

ř present: False

Vocabulary size: 36


Cell 6 — Save MMS V1.1 vocabulary

In [ ]:
# ============================================================
# CELL 6: Save MMS Corpus V1.1 vocabulary
# ============================================================

MMS_VOCAB_DIR.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    MMS_VOCAB_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        vocab_dict,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:")
print(MMS_VOCAB_PATH)

Saved:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/mms_tokenizer_v1_1/vocab.json


Cell 7 — Create Corpus V1.1 processor

In [ ]:
# ============================================================
# CELL 7: Create Corpus V1.1 MMS processor
# ============================================================

from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor
)

tokenizer = Wav2Vec2CTCTokenizer(
    str(MMS_VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

print("Tokenizer size:", len(tokenizer))
print("ř in tokenizer:", "ř" in tokenizer.get_vocab())

Tokenizer size: 38
ř in tokenizer: False


Cell 8 — Define MMS preprocessing

In [ ]:
# ============================================================
# CELL 8: Define MMS Corpus V1.1 preprocessing
# ============================================================

import soundfile as sf

def prepare_mms_dataset(example):

    audio_file = (
        PROJECT_ROOT
        / example["audio_path"]
    )

    audio, sampling_rate = sf.read(
        audio_file
    )

    inputs = processor(
        audio,
        sampling_rate=sampling_rate
    )

    example["input_values"] = (
        inputs.input_values[0]
    )

    example["input_length"] = len(
        example["input_values"]
    )

    example["labels"] = tokenizer(
        example["transcription"]
    ).input_ids

    return example


print("MMS preprocessing function ready.")

MMS preprocessing function ready.


Cell 9 — Test preprocessing on one segment

In [ ]:
# ============================================================
# CELL 9: Test MMS preprocessing
# ============================================================

sample = prepare_mms_dataset(
    dataset["train"][0]
)

print("Input samples:")
print(len(sample["input_values"]))

print("\nReference:")
print(sample["transcription"])

print("\nDecoded labels:")
print(
    tokenizer.decode(
        sample["labels"],
        group_tokens=False
    )
)

Input samples:
60160

Reference:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta

Decoded labels:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta


Cell 10 — Preprocess the full dataset

In [ ]:
# ============================================================
# CELL 10: Preprocess full MMS Corpus V1.1
# ============================================================

mms_dataset = dataset.map(
    prepare_mms_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=1
)

print(mms_dataset)

Map:   0%|          | 0/1472 [00:00<?, ? examples/s]

Map:   0%|          | 0/133 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 1472
    })
    validation: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 133
    })
})


Cell 11 — Save preprocessed MMS dataset

In [ ]:
# ============================================================
# CELL 11: Save preprocessed MMS Corpus V1.1
# ============================================================

mms_dataset.save_to_disk(
    str(MMS_DATASET_PATH)
)

print("Saved to:")
print(MMS_DATASET_PATH)

Cell 12 — Define CTC collator

In [ ]:
# ============================================================
# CELL 12: Define MMS CTC data collator
# ============================================================

from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch


@dataclass
class DataCollatorCTCWithPadding:

    processor: Any
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[
            Dict[
                str,
                Union[List[int], torch.Tensor]
            ]
        ]
    ) -> Dict[str, torch.Tensor]:

        input_features = [
            {
                "input_values":
                f["input_values"]
            }
            for f in features
        ]

        label_features = [
            {
                "input_ids":
                f["labels"]
            }
            for f in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels_batch = (
            self.processor.tokenizer.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt"
            )
        )

        labels = (
            labels_batch["input_ids"]
            .masked_fill(
                labels_batch.attention_mask.ne(1),
                -100
            )
        )

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor
)

print("MMS collator ready.")

Cell 13 — Define WER/CER

In [ ]:
# ============================================================
# CELL 13: Define MMS fine-tuning WER and CER
# ============================================================

import numpy as np
from jiwer import wer, cer


def compute_metrics(pred):

    pred_ids = np.argmax(
        pred.predictions,
        axis=-1
    )

    label_ids = pred.label_ids.copy()

    label_ids[label_ids == -100] = (
        processor.tokenizer.pad_token_id
    )

    pred_str = processor.batch_decode(
        pred_ids
    )

    label_str = processor.batch_decode(
        label_ids,
        group_tokens=False
    )

    return {
        "wer": wer(
            label_str,
            pred_str
        ) * 100,

        "cer": cer(
            label_str,
            pred_str
        ) * 100
    }


print("Metrics ready.")

Cell 14 — Load Tarifit-adapted MMS model with new output head

In [ ]:
# ============================================================
# CELL 14: Load Tarifit-adapted MMS model for Corpus V1.1
# ============================================================

from transformers import Wav2Vec2ForCTC

BASE_MMS_MODEL = (
    "iukocha/mms-tachebdant-from-tarifit"
)

model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MMS_MODEL,

    # Resize old MMS vocabulary to Corpus V1.1 vocabulary
    vocab_size=len(tokenizer),

    pad_token_id=tokenizer.pad_token_id,

    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,

    ignore_mismatched_sizes=True
)

print("Model loaded:", BASE_MMS_MODEL)

print(
    "Model output vocabulary:",
    model.config.vocab_size
)

print(
    "Tokenizer vocabulary:",
    len(tokenizer)
)

print(
    "Parameters:",
    round(
        model.num_parameters() / 1e6,
        1
    ),
    "M"
)

Cell 15 — Freeze convolutional feature encoder

In [ ]:
# ============================================================
# CELL 15: Freeze MMS convolutional feature encoder
# ============================================================

model.freeze_feature_encoder()

print("Feature encoder frozen.")

Cell 16 — Check GPU

In [ ]:
# ============================================================
# CELL 16: Check GPU before MMS fine-tuning
# ============================================================

import torch

print(
    "CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    free, total = torch.cuda.mem_get_info()

    print(
        "Total GPU memory:",
        round(total / 1024**3, 2),
        "GB"
    )

    print(
        "Free GPU memory:",
        round(free / 1024**3, 2),
        "GB"
    )